In [1]:
# Code for cross validation and splitting training data and test data
# IMPORTANT!!! The python version used for this code is 3.13.5, you need this exact version to get the same results 

In [2]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
import random

In [3]:
df = pd.read_csv("metadata.csv")        
df = df.sample(frac=1, random_state=40).reset_index(drop=True)      # Shuffle data

In [4]:
# IMPORTANT!!!! To check that the code is reproducible, you should get the exact order of images shown below (after shuffliing)
df.head(10)

,Unnamed: 0.1,Unnamed: 0,patient_id,lesion_id,smoke,drink,background_father,background_mother,age,pesticide,...,diagnostic,itch,grew,hurt,changed,bleed,elevation,img_id,biopsed,group_id
0,802,891,PAT_934,1778,False,False,POMERANIA,POMERANIA,68,True,...,SCC,False,UNK,False,UNK,False,False,PAT_934_1778_140.png,True,E
1,93,95,PAT_430,848,False,False,GERMANY,GERMANY,75,False,...,BCC,True,True,False,False,True,True,PAT_430_848_476.png,True,B
2,1469,1599,PAT_459,892,False,True,GERMANY,GERMANY,84,False,...,BCC,True,True,True,False,True,True,PAT_459_892_804.png,True,C
3,947,1039,PAT_753,1833,False,False,GERMANY,GERMANY,59,False,...,ACK,True,False,False,False,False,False,PAT_753_1833_710.png,False,C
4,1408,1537,PAT_320,681,False,True,ITALY,GERMANY,70,True,...,MEL,False,True,False,True,False,False,PAT_320_681_724.png,True,M
5,307,320,PAT_1392,1353,NaN,NaN,NaN,NaN,68,NaN,...,ACK,True,False,False,False,False,False,PAT_1392_1353_161.png,False,J
6,482,505,PAT_1197,713,NaN,NaN,NaN,NaN,54,NaN,...,NEV,False,True,False,False,False,False,PAT_1197_713_520.png,False,I
7,1107,1215,PAT_565,1073,True,True,POMERANIA,POMERANIA,51,True,...,BCC,True,False,False,False,False,False,PAT_565_1073_749.png,True,I
8,997,1090,PAT_343,717,False,False,GERMANY,GERMANY,21,False,...,SEK,False,UNK,False,UNK,False,True,PAT_343_717_927.png,True,N
9,700,787,PAT_184,284,False,False,ITALY,ITALY,71,False,...,ACK,False,True,False,False,False,True,PAT_184_284_721.png,True,C


In [5]:
# The method below uses cross-validation method for splitting training and testing data (basically splitting 5 times)
# The method ensures that alomst 80% of EACH diagnosis is in the training data, and 20% in the test data 
# The method ensures that all images belonging to the same patient are in either the training group or test group

sgkf = StratifiedGroupKFold().split(X=df, y=df["diagnostic"], groups=df["patient_id"])      # default n_splits=5
all_splits= []
for i, j in enumerate(sgkf):
    all_splits.append(j)

# For each split_n, split_n[0] is for the training data, and split_n[0] is for the test data
# These only include the iindices for the data
split_0 = all_splits[0]
split_1 = all_splits[1]
split_2 = all_splits[2]
split_3 = all_splits[3]
split_4 = all_splits[4]

In [6]:
df.loc[split_0[1], "group_id"] = "a"
df.loc[split_1[1], "group_id"] = "b"
df.loc[split_2[1], "group_id"] = "c"
df.loc[split_3[1], "group_id"] = "d"
df.loc[split_4[1], "group_id"] = "e"

In [7]:
df.to_csv("data_with_splits.csv")

**THE REST BELOW IS JUST FOR CHECKING ACCURACY OF CODE**

In [8]:
# Checking that there is no data leakage
print( 
set(df.iloc[split_0[0]]["patient_id"]).isdisjoint(set(df.iloc[split_0[1]]["patient_id"])), 
set(df.iloc[split_1[0]]["patient_id"]).isdisjoint(set(df.iloc[split_1[1]]["patient_id"])), 
set(df.iloc[split_2[0]]["patient_id"]).isdisjoint(set(df.iloc[split_2[1]]["patient_id"])), 
set(df.iloc[split_3[0]]["patient_id"]).isdisjoint(set(df.iloc[split_3[1]]["patient_id"])), 
set(df.iloc[split_4[0]]["patient_id"]).isdisjoint(set(df.iloc[split_4[1]]["patient_id"])) 
)

True True True True True


In [9]:
# Checking stratification of data
for disease in df["diagnostic"].unique():
    data0 = df.iloc[split_0[0]]         # simply change [0] to [1] for checking the test data
    data1 = df.iloc[split_1[0]]
    data2 = df.iloc[split_2[0]]
    data3 = df.iloc[split_3[0]]
    data4 = df.iloc[split_4[0]]
    print(f"{disease}, {round(len(df[df["diagnostic"] == disease])/len(df), 3)}")
    
    print(round(len(data0[data0["diagnostic"] == disease])/len(data0), 3))
    print(round(len(data1[data1["diagnostic"] == disease])/len(data1), 3))
    print(round(len(data2[data2["diagnostic"] == disease])/len(data2), 3))
    print(round(len(data3[data3["diagnostic"] == disease])/len(data3), 3))
    print(round(len(data4[data4["diagnostic"] == disease])/len(data4), 3))
    print("\n")

SCC, 0.087
0.088
0.087
0.087
0.087
0.087


BCC, 0.388
0.388
0.387
0.387
0.388
0.388


ACK, 0.301
0.301
0.302
0.301
0.301
0.301


MEL, 0.023
0.023
0.023
0.024
0.023
0.023


NEV, 0.105
0.105
0.105
0.105
0.105
0.105


SEK, 0.096
0.095
0.096
0.096
0.096
0.096




In [10]:
# Checking that 80% of each diagnostic is present in the training data 
for disease in df["diagnostic"].unique():
    data0 = df.iloc[split_0[0]]         # simply change [0] to [1] for checking the test data
    data1 = df.iloc[split_1[0]]
    data2 = df.iloc[split_2[0]]
    data3 = df.iloc[split_3[0]]
    data4 = df.iloc[split_4[0]]
    print(disease)
    print(round(len(data0[data0["diagnostic"] == disease])/len(df[df["diagnostic"] == disease]), 3))
    print(round(len(data1[data1["diagnostic"] == disease])/len(df[df["diagnostic"] == disease]), 3))
    print(round(len(data2[data2["diagnostic"] == disease])/len(df[df["diagnostic"] == disease]), 3))
    print(round(len(data3[data3["diagnostic"] == disease])/len(df[df["diagnostic"] == disease]), 3))
    print(round(len(data4[data4["diagnostic"] == disease])/len(df[df["diagnostic"] == disease]), 3))
    print("\n")

SCC
0.804
0.799
0.799
0.799
0.799


BCC
0.8
0.8
0.8
0.8
0.8


ACK
0.8
0.801


0.8
0.8
0.8


MEL
0.796
0.796
0.816
0.796
0.796


NEV
0.8
0.8
0.8
0.8
0.8


SEK
0.796
0.801
0.801
0.801
0.801


